# Overview of the Eval Decoders

The BioJEPA-AC model is an encoder, meaning its output is an embedding representation, not a specific molecular representation or property. To see the details, you can review the [AC Predictor](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_ac_predictor_v1_0.ipynb). To complete benchmarks against other models, we need to add a decoder head on top.

We do need to be careful here, though. Our goal with BioJEPA is to build a great generalizable foundation model. Because of this, we need to make sure our head isn't so smart it masks issues in it. Because of this, you'll see our heads are shallow.

We have 2 heads for our v1.0 evals:
1. Linear expression decoder
2. Linear classifier

![Eval Decoder Overview](../resources/v1_0/eval_decoder_overview.png)


**Linear Expression Decoder**

For Perturb-seq, a common eval is predicting expression values or changes in expression. For our model to do this, we'll have to project from our latent space down to a single value per gene. Since our cell state latent representation is already $[B, \text{n\_genes}, \text{embed\_dim}]$, this decoder will use a single linear layer to pull the representation down to a single value per gene.

**Linear Classifier**

BioJEPA has a lot of information in its representation of cells in the latent space. As part of our evals, we review how we can extract cell batch information, cell types, whether a cell is perturbed, perturbation modes, and perturbation pathways. To evaluate input properties from the latent representation, we use a classifier head on top of both the model and the submodules. For cell state latents, we first use mean pooling across the genes to get a single representation per cell. We then use a linear layer to project from the embedding dimensions to the classifier dimensions.

This notebook will walk through each decoder we use so that the reader can build an intuition for what each head is doing to the data. To this end, you'll see that we set the layer initializations and values to ones so you can calculate them by hand if you need to follow a layer more closely.

In [1]:
import torch
import numpy as np
import torch.nn as nn

In [2]:
SEED = 1337
torch.manual_seed(SEED)
np.random.seed(SEED)

In [3]:
def mock_data(dim_1, dim_2, low=1, high=9):
    return torch.from_numpy(np.round(np.random.uniform(low, high, size=(dim_1, dim_2)), 0)).float() * 0.1

## Linear Expression Decoder

We'll start with the expression decoder. As mentioned, the goal of this decoder is to generate per-gene expression predictions. While the AC Predictor does output both a mean and logvar, in practice we'll just use the mean for our expression prediction and reserve the logvar for error analysis.

We'll start by generating the input. We'll use the same dimensions we used in other explainer notebooks but regenerate the data.

### Data Prep

We'll start with a simple data prep. The linear expression decoder takes just cell state latents as input. We'll have a batch of 2 to show how two different cells are processed.

In [4]:
batch = 2
num_genes = 8
embed_dim = 6

In [5]:
cell_latent = mock_data(batch * num_genes, embed_dim).reshape(batch, num_genes, embed_dim)
cell_latent.shape, cell_latent

(torch.Size([2, 8, 6]),
 tensor([[[0.3000, 0.2000, 0.3000, 0.5000, 0.4000, 0.5000],
          [0.3000, 0.9000, 0.7000, 0.2000, 0.4000, 0.6000],
          [0.2000, 0.9000, 0.5000, 0.7000, 0.7000, 0.4000],
          [0.4000, 0.6000, 0.7000, 0.3000, 0.3000, 0.6000],
          [0.5000, 0.2000, 0.4000, 0.3000, 0.5000, 0.8000],
          [0.2000, 0.9000, 0.4000, 0.5000, 0.5000, 0.1000],
          [0.7000, 0.9000, 0.2000, 0.2000, 0.4000, 0.7000],
          [0.1000, 0.4000, 0.1000, 0.7000, 0.4000, 0.7000]],
 
         [[0.5000, 0.9000, 0.8000, 0.6000, 0.6000, 0.9000],
          [0.3000, 0.1000, 0.2000, 0.9000, 0.9000, 0.5000],
          [0.4000, 0.7000, 0.1000, 0.7000, 0.6000, 0.3000],
          [0.6000, 0.8000, 0.2000, 0.8000, 0.4000, 0.7000],
          [0.3000, 0.5000, 0.2000, 0.5000, 0.5000, 0.4000],
          [0.7000, 0.5000, 0.7000, 0.7000, 0.4000, 0.8000],
          [0.7000, 0.8000, 0.4000, 0.5000, 0.8000, 0.6000],
          [0.5000, 0.3000, 0.9000, 0.5000, 0.7000, 0.3000]]]))

### Forward Pass

The goal of this model is to project the cell_latent down to per-gene expression, without covering up missing information in the latent. Since the genes in gene expression are our tokens, our task is simple: we just collapse the embedding dimension down to 1 to get a single value per gene. This value becomes our expression representation.

#### Linear Expression Layer

The linear layer does the projection from the embedding dimensions to 1 so that we have a single value per token (aka gene). To show how this aggregates across embedding dimensions, we'll do incremental weights so you'll see the impact of the collapsing.

In [6]:
lin_exp = nn.Linear(embed_dim, 1)
pattern = torch.arange(embed_dim).unsqueeze(0) * 1.0
lin_exp.weight = nn.Parameter(pattern)
nn.init.zeros_(lin_exp.bias)
lin_exp.weight, lin_exp.bias

(Parameter containing:
 tensor([[0., 1., 2., 3., 4., 5.]], requires_grad=True),
 Parameter containing:
 tensor([0.], requires_grad=True))

In [7]:
gene_preds = lin_exp(cell_latent)
gene_preds.shape, gene_preds

(torch.Size([2, 8, 1]),
 tensor([[[ 6.4000],
          [ 7.5000],
          [ 8.8000],
          [ 7.1000],
          [ 7.9000],
          [ 5.7000],
          [ 7.0000],
          [ 7.8000]],
 
         [[11.2000],
          [ 9.3000],
          [ 6.9000],
          [ 8.7000],
          [ 6.4000],
          [ 9.6000],
          [ 9.3000],
          [ 7.9000]]], grad_fn=<ViewBackward0>))

**Expression prediction per gene**

We will now remove the embedding dimension. This will give us one predicted expression value per gene.

In [8]:
gene_preds = gene_preds.squeeze(-1)
gene_preds.shape, gene_preds

(torch.Size([2, 8]),
 tensor([[ 6.4000,  7.5000,  8.8000,  7.1000,  7.9000,  5.7000,  7.0000,  7.8000],
         [11.2000,  9.3000,  6.9000,  8.7000,  6.4000,  9.6000,  9.3000,  7.9000]],
        grad_fn=<SqueezeBackward1>))

#### Expression Predictions

With just that single layer and reshape, we've now calculated one predicted expression value per gene. The decoder collapses each gene's embedding dimensions down to a single value while keeping the batch and gene dimensions intact.

## Linear Classifier

We'll now show how our linear classifier works. As mentioned, our linear classifier for evals is built to evaluate a number of input metadata fields, such as cell batch information, cell types, whether a cell is perturbed, perturbation modes, and perturbation pathways. The goal of our classifier is to generate a raw value, called a logit, for each class. As we dive into each eval, you'll see how the input and classes change.

We'll start by generating another input that represents a cell state latent, along with a number of classes. We'll use the same dimensions we used in other explainer notebooks but regenerate the data.

### Data Prep

We'll start with a simple data prep. We'll start with a cell state latent and mean pool across the genes to create a single representation per cell. We'll then give that pooled representation to the linear classifier. We'll need to preconfigure the number of classes and will use a batch of 2 to show how two different cells are processed.

In [9]:
batch = 2
num_genes = 8
embed_dim = 6
num_classes = 3

In [10]:
cell_latent = mock_data(batch * num_genes, embed_dim).reshape(batch, num_genes, embed_dim)
cell_latent.shape, cell_latent

(torch.Size([2, 8, 6]),
 tensor([[[0.4000, 0.4000, 0.5000, 0.2000, 0.6000, 0.1000],
          [0.3000, 0.7000, 0.8000, 0.6000, 0.4000, 0.8000],
          [0.9000, 0.3000, 0.4000, 0.7000, 0.4000, 0.1000],
          [0.4000, 0.5000, 0.1000, 0.8000, 0.7000, 0.7000],
          [0.2000, 0.1000, 0.3000, 0.2000, 0.3000, 0.6000],
          [0.5000, 0.8000, 0.6000, 0.2000, 0.2000, 0.4000],
          [0.4000, 0.2000, 0.6000, 0.1000, 0.7000, 0.6000],
          [0.1000, 0.7000, 0.4000, 0.7000, 0.4000, 0.6000]],
 
         [[0.2000, 0.8000, 0.3000, 0.7000, 0.6000, 0.8000],
          [0.5000, 0.1000, 0.2000, 0.9000, 0.5000, 0.2000],
          [0.6000, 0.6000, 0.5000, 0.8000, 0.8000, 0.3000],
          [0.7000, 0.5000, 0.3000, 0.1000, 0.3000, 0.6000],
          [0.3000, 0.5000, 0.4000, 0.5000, 0.6000, 0.9000],
          [0.1000, 0.5000, 0.6000, 0.2000, 0.7000, 0.4000],
          [0.5000, 0.6000, 0.8000, 0.7000, 0.6000, 0.1000],
          [0.1000, 0.5000, 0.7000, 0.7000, 0.3000, 0.3000]]]))

### Forward Pass

The goal of this model is to project the input latent (cell_latent) down to per-class values per input example, without covering up missing information in the latent.

Our input cell latent has $[\text{batch},\text{n\_genes},\text{embed\_dim}]$, so we'll first use mean pooling across the genes to get a single representation per cell with shape $[\text{batch},\text{embed\_dim}]$. We'll then use a linear layer to project that representation down to $[\text{batch},\text{class}]$.

#### (Optional) Mean Pooling

Since our input has one latent representation per gene, we'll first collapse the gene dimension down to a single representation per cell. We do this by taking the mean across the genes for each embedding dimension.

*Note that this step can be skipped if the input already has 2 dimensions, like in the case of classifiers that we run on top of the Action Composer output.*

In [11]:
cell_embedding = cell_latent.mean(dim=1)
cell_embedding.shape, cell_embedding

(torch.Size([2, 6]),
 tensor([[0.4000, 0.4625, 0.4625, 0.4375, 0.4625, 0.4875],
         [0.3750, 0.5125, 0.4750, 0.5750, 0.5500, 0.4500]]))

#### Linear Classifier Layer

The linear layer does the projection from the pooled embedding dimensions to our number of classes so that we have a per-class value for each example. To show how this aggregates across embedding dimensions, we'll do incremental weights so you'll see the impact of the collapsing.

In [12]:
class_pred = nn.Linear(embed_dim, num_classes)
vs, d = embed_dim, num_classes
rows = torch.full((vs,), 0.1).unsqueeze(0)
cols = torch.arange(d).unsqueeze(1)
pattern = 1 * (rows + 0.1 * cols)
class_pred.weight = nn.Parameter(pattern)
nn.init.zeros_(class_pred.bias)
class_pred.weight, class_pred.bias

(Parameter containing:
 tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
         [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
         [0.3000, 0.3000, 0.3000, 0.3000, 0.3000, 0.3000]], requires_grad=True),
 Parameter containing:
 tensor([0., 0., 0.], requires_grad=True))

In [13]:
lin_class = class_pred(cell_embedding)
lin_class.shape, lin_class

(torch.Size([2, 3]),
 tensor([[0.2713, 0.5425, 0.8138],
         [0.2938, 0.5875, 0.8813]], grad_fn=<AddmmBackward0>))

#### Class Logits

With just mean pooling and a single linear layer, we've gone from our input into per-example class logits. These values aren't expected to add up to 1 because they haven't been converted into probabilities. Some of our evals will pick the highest logit when we need a class, while others will use softmax to convert the logits into probabilities for metrics like AUROC. By keeping our classifier generic and outputting raw values, we create a flexible class we can use across our different evals.